# Parakeet ASR on a free Colab GPU → OpenShorts

Hosts **NVIDIA parakeet-tdt-0.6b-v3** (q8_0 GGUF) with **NeMo-Speech.cpp** —
NVIDIA's official prebuilt runtime, no compilation — on a free Colab T4.

**The model loads ONCE on the GPU and stays resident** (nemo-speech serve),
so every transcription answers in seconds instead of reloading weights.
A Cloudflare quick tunnel publishes it, and the notebook prints exactly the
two values to paste into OpenShorts: **Settings → transcription models →
Google Colab URL + API key**.

> Runtime → Change runtime type → **T4 GPU** before running. Free Colab
> disconnects after ~12h and every re-run gets a new URL — paste the new one
> when that happens (the model download is cached for the session, so a
> re-run is fast until the session itself dies).

In [ ]:
# 1) GPU check — fail loudly if no GPU is attached
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert out.returncode == 0 and out.stdout.strip(), "No NVIDIA GPU visible - Colab: Runtime > Change runtime type > T4 GPU"
print("GPU:", out.stdout.strip())

In [ ]:
# 2) Prebuilt NeMo-Speech.cpp runtime — download, verify sha256, extract
# (idempotent: skipped when the binary already exists)
import os, subprocess, urllib.request, hashlib

VERSION = "0.3.0"
BASE = f"https://github.com/NVIDIA/NeMo-Speech.cpp/releases/download/v{VERSION}"
ARCHIVE = f"/content/nemo-speech-{VERSION}-linux-x86_64-cuda.tar.gz"
BIN = "/opt/nemo-speech/bin/nemo-speech"

if not os.path.exists(BIN):
    urllib.request.urlretrieve(f"{BASE}/nemo-speech-{VERSION}-linux-x86_64-cuda.tar.gz", ARCHIVE)
    urllib.request.urlretrieve(f"{BASE}/nemo-speech-{VERSION}-linux-x86_64-cuda.tar.gz.sha256", ARCHIVE + ".sha256")
    expected = open(ARCHIVE + ".sha256").read().split()[0]
    actual = hashlib.sha256(open(ARCHIVE, "rb").read()).hexdigest()
    assert actual == expected, f"checksum mismatch: {actual} != {expected}"
    print("checksum verified:", expected[:16] + "...")
    os.makedirs("/opt", exist_ok=True)
    subprocess.run(["tar", "-xzf", ARCHIVE, "-C", "/opt"], check=True)
    subprocess.run(["ln", "-sf", subprocess.run(["find", "/opt/nemo-speech", "-type", "f", "-name", "nemo-speech"],
                  capture_output=True, text=True).stdout.splitlines()[0], "/usr/local/bin/nemo-speech"], check=True)
else:
    print("runtime already extracted")

print(subprocess.run([BIN, "--version"], capture_output=True, text=True).stdout.strip())

In [ ]:
# 3) Pull the model (714 MB q8_0 GGUF, sha256-verified by the CLI, idempotent)
import subprocess
print(subprocess.run(["/opt/nemo-speech/bin/nemo-speech", "pull", "parakeet-tdt"],
                     capture_output=True, text=True).stdout.strip() or "model already cached")

In [ ]:
# 4) Start the RESIDENT model server (loads once on the GPU, stays loaded)
# The API key protects the public tunnel - OpenShorts sends it as a Bearer token.
import secrets, subprocess, time, urllib.request

API_KEY = secrets.token_hex(16)
serve_log = open("/content/serve.log", "w")
proc = subprocess.Popen(
    ["/opt/nemo-speech/bin/nemo-speech", "serve",
     "--asr-model", "parakeet-tdt",
     "--host", "0.0.0.0", "--port", "8000",
     "--api-key", API_KEY],
    stdout=serve_log, stderr=subprocess.STDOUT, start_new_session=True)

for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=3) as r:
            if r.status == 200:
                break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("server did not come up - check /content/serve.log")

print(open("/content/serve.log").read())
assert "backend=CUDA" in open("/content/serve.log").read(), "model is NOT on the GPU - check the log"
print("Model resident on GPU, server healthy.")

In [ ]:
# 5) Cloudflare quick tunnel -> public URL for OpenShorts
# (idempotent: reuses the downloaded cloudflared binary)
import os, re, subprocess, time, urllib.request

CF = "/content/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, 0o755)

tunnel_log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen([CF, "tunnel", "--url", "http://localhost:8000"],
                          stdout=tunnel_log, stderr=subprocess.STDOUT,
                          start_new_session=True)

url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0)
        break
assert url, "tunnel URL not found - check /content/tunnel.log"
print("=" * 60)
print("COLAB URL:", url)
print("API KEY: ", API_KEY)
print("=" * 60)
print("Paste both into OpenShorts:")
print("  Settings > transcription models > Google Colab URL + API key")
print("Then hit 'test colab' in that card.")

## Keep-alive / monitor

The server and tunnel keep running while this notebook is open. Re-run cell 5
if the tunnel restarts, and re-run cell 4 only if the server itself died
(check `/content/serve.log`).

Free Colab disconnects after ~12h; every new session re-runs everything from
cell 1 (the runtime/model downloads repeat but the tunnel URL always changes —
update OpenShorts when it does).

Reminder: a persistent server bends Colab's acceptable-use rules — this is
for personal use.